In [ ]:
import torch

print(f"PyTorch 버전: {torch.__version__}")
print(f"CUDA 사용 가능 여부: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"사용 중인 GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# 데이터 처리
import pandas as pd
import numpy as np

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns

# 텍스트 탐색용
import re
from collections import Counter

# 머신러닝 (scikit-learn)
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

# 모델
from xgboost import XGBClassifier

# 희소 행렬(sparse matrix) 타입 확인용
from scipy import sparse

# 그래프에서 한글이 깨지지 않도록 폰트 설정 (Windows 기준: 맑은 고딕)
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False  # 마이너스 기호 깨짐 방지

#plt 그래프 스타일 설정
plt.style.use("ggplot")

# 노트북 안에서 그래프가 바로 보이도록 설정
%matplotlib inline

print("라이브러리 불러오기 완료")

In [ ]:
# 위에서 정의한 경로 변수를 사용해 세 개의 데이터를 불러옵니다.
# train: 모델이 학습할 로그 텍스트(full_log)와 정답 라벨(level)이 들어 있습니다.
# test : 예측 대상 데이터입니다. 정답 level이 없으므로 우리가 예측해서 채웁니다.
# sample_submission: 제출 파일의 형식(행 개수, 컬럼)을 알려주는 양식입니다.
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

# 각 데이터의 행/열 개수를 확인합니다. (train과 test의 행 개수는 보통 다릅니다.)
print("train shape:", train.shape)
print("test shape :", test.shape)
print("sample_submission shape:", sample_submission.shape)

In [ ]:
# ---- 파일 경로 설정 ----
# 노트북과 같은 폴더에 있는 파일명을 기준으로 경로를 설정합니다.
# 파일을 다른 폴더에 저장했거나 파일명이 다르면, 아래 값을 실제 파일 위치에 맞게 수정합니다.
# 이후 코드들은 파일명을 직접 쓰지 않고 이 변수들을 사용하므로, 경로 변경은 이 셀만 수정하면 됩니다.
TRAIN_PATH = "security_log_train.csv"             # 학습 데이터 (로그 텍스트 + 정답 level)
TEST_PATH = "security_log_test.csv"               # 테스트 데이터 (정답 level 없음)
SAMPLE_SUBMISSION_PATH = "sample_submission.csv"  # 제출 양식
OUTPUT_PATH = "submission.csv"                     # 생성할 제출 파일

# ---- 실행 옵션 ----
# RANDOM_STATE: 결과 재현성을 위한 난수 시드입니다.
# 같은 시드를 사용하면 train/valid 분리나 모델 학습 결과가 매번 동일하게 재현됩니다.
RANDOM_STATE = 42

print("설정 완료 (전체 train 데이터를 사용합니다)")

In [ ]:
# ============================================================
# 모델 학습에 사용할 데이터 준비
# ============================================================

# 이 baseline에서는 전체 train 데이터를 사용합니다.
# 클래스 불균형이 매우 심한 데이터이므로,
# 일부 샘플만 뽑으면 희귀 클래스가 사라지거나
# train/validation 분리 과정에서 오류가 발생할 수 있습니다.
#
# 따라서 교육용 baseline에서는 샘플링하지 않고 전체 train을 사용합니다.

train_model = train.copy()

print("모델 학습에 사용할 train_model 크기:", train_model.shape)

print("\n[train_model level 분포]")
print(train_model["level"].value_counts().sort_index())

In [ ]:
# 검증 단계에서는 데이터의 80%만 학습에 사용했습니다.
# 제출용 모델은 가지고 있는 학습 데이터를 모두 사용해 다시 학습하는 것이 일반적으로 더 좋습니다.

# 1) 최종 학습에 사용할 전체 입력/정답 (train_model 전체)
X_all = train_model["full_log"]
y_all = train_model["level"]

# 2) TF-IDF를 전체 학습 데이터로 다시 fit 합니다. (검증 때와 동일한 설정)
final_tfidf = TfidfVectorizer(
    max_features=5000,     # 사용할 단어(feature) 개수를 5000개로 제한 (baseline 기준)
    ngram_range=(1, 2),    # 단어 1개(unigram) + 2개 묶음(bigram)까지 사용
    min_df=3,              # 너무 드문 단어(3개 미만 문서에만 등장)는 제외
    max_df=0.95, 
)
# 학습 데이터는 fit_transform으로 단어 사전을 만들며 변환합니다.
X_all_tfidf = final_tfidf.fit_transform(X_all)
# 테스트 데이터는 transform만 사용합니다. fit_transform을 쓰면 데이터 누수가 되므로 주의합니다.
X_test_tfidf = final_tfidf.transform(test["full_log"])

X_all_tfidf = X_all_tfidf.astype('float32')
X_test_tfidf = X_test_tfidf.astype('float32')

print("X_all_tfidf  shape:", X_all_tfidf.shape)
print("X_test_tfidf shape:", X_test_tfidf.shape)

In [ ]:
# 3) 최종 XGBoost 모델을 전체 학습 데이터로 학습합니다. (검증 때와 동일한 설정)
final_model = XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    num_class=num_classes,
    eval_metric="mlogloss",
    random_state=RANDOM_STATE,
    tree_method="hist",         # GPU 가속을 위한 트리 빌드 방법
    device="cuda",              # RTX A4000 GPU 지정을 위한 설정
    # n_jobs=-1,
)
final_model.fit(X_all_tfidf, y_all)

# 4) 테스트 데이터를 예측합니다.
#    predict는 각 샘플에 대해 가장 확률이 높은 클래스(level 0~6 중 하나)를 바로 돌려줍니다.
#    (각 클래스의 확률값 자체가 필요할 때는 뒤의 선택 실험에서 predict_proba를 사용합니다.)
#    이 기본 예측값은 test_pred_basic 으로 따로 두어, 뒤의 선택 실험 결과와 구분합니다.
test_pred_basic = final_model.predict(X_test_tfidf)
print("최종 모델 학습 및 test 예측 완료")
print("예측 결과 개수:", len(test_pred_basic))
print("예측된 level 분포:", pd.Series(test_pred_basic).value_counts().sort_index().to_dict())

In [ ]:
# ============================================================
# 선택 실험: 예측 확률이 낮은 샘플을 level 7로 변경하기
# 이 셀은 기본 학습이 아니라 선택 후처리 실험입니다. 기본 제출과 구분합니다.
# ============================================================

# predict_proba는 각 샘플이 클래스(0~6)에 속할 예측 확률을 모두 돌려줍니다.
# (predict는 이 중 가장 큰 확률의 클래스 하나만 돌려줍니다.)
test_proba = final_model.predict_proba(X_test_tfidf)

# 각 샘플에서 가장 높은 클래스 확률을 구합니다.
# 이 값은 모델이 자신의 예측을 얼마나 확신하는지 확인하는 데 사용합니다.
max_proba = np.max(test_proba, axis=1)

# THRESHOLD: 기존 예측을 유지할 최소 확신 기준입니다.
# 최대 예측 확률이 이 값보다 낮은 샘플은 모델이 충분히 확신하지 못한 것으로 보고 level 7로 바꿉니다.
# 주의: 0.5는 정답값이 아니라 실험 시작값입니다. (threshold 값에 따라 결과가 달라집니다.)
THRESHOLD = 0.5

# 기본 예측을 복사한 뒤, 확신도가 낮은 샘플만 level 7로 바꿉니다. (기본 예측값은 그대로 보존)
test_pred_with_7 = test_pred_basic.copy()
test_pred_with_7[max_proba < THRESHOLD] = 7

print("[기본 예측 분포]")
print(pd.Series(test_pred_basic).value_counts().sort_index())

print("\n[level 7 후처리 적용 후 예측 분포]")
print(pd.Series(test_pred_with_7).value_counts().sort_index())

print("\nlevel 7로 변경된 개수:", (test_pred_with_7 == 7).sum())
print("전체 test 개수:", len(test_pred_with_7))
print("level 7 비율:", (test_pred_with_7 == 7).mean())

In [ ]:
# ============================================================
# threshold 값에 따른 level 7 예측 개수 비교
# ============================================================

threshold_list = [0.5, 0.6, 0.7, 0.8, 0.9]

threshold_summary = []

for threshold in threshold_list:
    temp_pred = test_pred_basic.copy()
    temp_pred[max_proba < threshold] = 7

    threshold_summary.append({
        "threshold": threshold,
        "level_7_count": int((temp_pred == 7).sum()),
        "level_7_ratio": float((temp_pred == 7).mean()),
    })

threshold_summary = pd.DataFrame(threshold_summary)

display(threshold_summary)

In [ ]:
# ============================================================
# 제출에 사용할 예측값 선택
# ============================================================

# 기본 baseline 제출값: level 0~6만 예측
test_pred_for_submission = test_pred_basic

# level 7 후처리를 적용하고 싶으면 아래 줄의 주석을 해제하고,
# 위 줄을 주석 처리하세요.
# test_pred_for_submission = test_pred_with_7

print("최종 제출에 사용할 예측값 분포:")
print(pd.Series(test_pred_for_submission).value_counts().sort_index())

In [ ]:
# sample_submission을 복사해서 제출 형식(행 개수, 컬럼)을 그대로 유지합니다.
submission = sample_submission.copy()

# sample_submission 의 컬럼 구조를 먼저 확인합니다.
print("sample_submission 컬럼:", sample_submission.columns.tolist())

# 일반적으로 sample_submission의 "마지막 컬럼"이 예측값을 넣는 컬럼입니다.
# 이 대회의 sample_submission은 ['id', 'level'] 구조라서 마지막 컬럼이 'level'입니다.
# columns[-1] 방식은 컬럼명이 달라도 동작하도록 일반화한 것이며, 대회마다 컬럼명을 직접 확인합니다.
target_col = submission.columns[-1]
print("예측값을 넣을 컬럼:", target_col)

# sample_submission의 마지막 컬럼에 어떤 값들이 들어있는지 예시로 확인합니다.
# (단, sample_submission만으로 모든 허용 label을 알 수는 없습니다. 대회 설명을 확인하세요.)
print("sample_submission 마지막 컬럼 고유값 예시:")
print(sample_submission[target_col].unique()[:20])

# 행 개수가 test 데이터와 같은지 확인합니다. (제출 파일은 test와 행 개수가 같아야 합니다.)
print("test 행 개수:", len(test))
print("submission 행 개수:", len(submission))

# 예측 결과를 제출 파일의 예측 컬럼(level)에 저장합니다. (반드시 test_pred_for_submission 사용)
submission[target_col] = test_pred_for_submission

# 안전 확인: 제출 파일과 test의 행 개수가 같아야 합니다.
assert len(submission) == len(test), "submission과 test의 행 개수가 다릅니다."

# 제출 예측값 분포 확인 (특정 클래스만 과도하게 예측되는지 점검)
print("\n제출 예측값 분포:")
print(submission[target_col].value_counts().sort_index())

# 저장 (한글/엑셀 호환을 위해 utf-8-sig)
submission.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

# 저장된 CSV 파일을 데이콘 제출 페이지에 업로드하면 됩니다.
print(f"\n제출 파일이 저장되었습니다: {OUTPUT_PATH}")
display(submission.head())